In [1]:
import os
import pandas as pd
from datetime import date, timedelta
from pandas.tseries.offsets import *
from sqlalchemy import create_engine

data_path = "../data/"
xlx_path = "../Excel/"
osd_path = "\\Users\\User\\OneDrive\\Documents\\obsidian-git-sync\\Data\\"

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 2, 1), datetime.date(2023, 1, 31))

In [2]:
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(1)
yesterday = today - num_business_days
print(f'today: {today}')
yesterday = yesterday.date()
print(f'yesterday: {yesterday}')

today: 2023-02-01
yesterday: 2023-01-31


In [3]:
file_name = "StockQuotation.xlsx"
input_file = xlx_path + file_name
df_S100 = pd.read_excel(input_file, header=8)
df_S100.shape

(100, 16)

In [4]:
file_name = "StockQuotation (1).xlsx"
input_file = xlx_path + file_name
df_sSET = pd.read_excel(input_file, header=8)
df_sSET.shape

(144, 16)

In [5]:
file_name = "StockQuotation (2).xlsx"
input_file = xlx_path + file_name
df_CLMV = pd.read_excel(input_file, header=8)
df_CLMV.shape

(57, 16)

In [6]:
file_name = "StockQuotation (3).xlsx"
input_file = xlx_path + file_name
df_HD = pd.read_excel(input_file, header=8)
df_HD.shape

(30, 16)

In [7]:
df = pd.concat([df_S100, df_sSET, df_CLMV, df_HD])
df.shape

(331, 16)

In [8]:
df.dtypes

Symbol              object
Trading Sign        object
Last               float64
Change             float64
%Change            float64
Open               float64
High               float64
Low                float64
TR Volume            int64
TR Value (k)       float64
Total Volume         int64
Total Value (k)    float64
Bid Size             int64
Bid                float64
Offer              float64
Offer Size           int64
dtype: object

In [9]:
### First equals to latest published pdf file
df = df.drop_duplicates(subset='Symbol', keep='first')
df.shape

(250, 16)

In [10]:
df.rename(columns={col: col.replace(" ", "-") for col in df.columns}, inplace=True)
cols = 'Symbol Last High Low Total-Volume Open'.split()
df[cols]

,Symbol,Last,High,Low,Total-Volume,Open
0,AAV,3.08,3.12,3.04,59242050,3.12
1,ACE,2.58,2.60,2.56,13932246,2.58
2,ADVANC,198.00,198.00,196.00,2711705,196.00
3,AMATA,20.10,20.20,20.00,2118401,20.10
4,AOT,74.25,74.75,74.00,9675814,74.75
...,...,...,...,...,...,...
12,BPP,17.20,17.40,17.10,542615,17.30
27,JWD,21.40,21.60,21.20,2099125,21.50
45,SCCC,160.00,160.50,159.50,33591,159.50
50,SUPER,0.64,0.64,0.63,3721509,0.64


In [11]:
df.insert(1, 'Date', '2023-02-01')
colt = 'Symbol Date Last High Low Total-Volume Open'.split()
df[colt]

,Symbol,Date,Last,High,Low,Total-Volume,Open
0,AAV,2023-02-01,3.08,3.12,3.04,59242050,3.12
1,ACE,2023-02-01,2.58,2.60,2.56,13932246,2.58
2,ADVANC,2023-02-01,198.00,198.00,196.00,2711705,196.00
3,AMATA,2023-02-01,20.10,20.20,20.00,2118401,20.10
4,AOT,2023-02-01,74.25,74.75,74.00,9675814,74.75
...,...,...,...,...,...,...,...
12,BPP,2023-02-01,17.20,17.40,17.10,542615,17.30
27,JWD,2023-02-01,21.40,21.60,21.20,2099125,21.50
45,SCCC,2023-02-01,160.00,160.50,159.50,33591,159.50
50,SUPER,2023-02-01,0.64,0.64,0.63,3721509,0.64


In [12]:
file_name = "name-ttl.csv"
input_file = data_path + file_name
input_file 

'../data/name-ttl.csv'

In [13]:
df_name = pd.read_csv(input_file, header=None, names=['Symbol'])
df_name

,Symbol
0,ACE
1,ADVANC
2,AEONTS
3,AH
4,AIE
...,...
228,WHAIR
229,WHART
230,WHAUP
231,WICE


In [14]:
df_out = df[colt]
df_out

,Symbol,Date,Last,High,Low,Total-Volume,Open
0,AAV,2023-02-01,3.08,3.12,3.04,59242050,3.12
1,ACE,2023-02-01,2.58,2.60,2.56,13932246,2.58
2,ADVANC,2023-02-01,198.00,198.00,196.00,2711705,196.00
3,AMATA,2023-02-01,20.10,20.20,20.00,2118401,20.10
4,AOT,2023-02-01,74.25,74.75,74.00,9675814,74.75
...,...,...,...,...,...,...,...
12,BPP,2023-02-01,17.20,17.40,17.10,542615,17.30
27,JWD,2023-02-01,21.40,21.60,21.20,2099125,21.50
45,SCCC,2023-02-01,160.00,160.50,159.50,33591,159.50
50,SUPER,2023-02-01,0.64,0.64,0.63,3721509,0.64


In [17]:
df_merge = pd.merge(df_name, df_out, on = 'Symbol', how='inner')
df_merge.head()

,Symbol,Date,Last,High,Low,Total-Volume,Open
0,ACE,2023-02-01,2.58,2.60,2.56,13932246,2.58
1,ADVANC,2023-02-01,198.00,198.00,196.00,2711705,196.00
2,AH,2023-02-01,33.25,33.50,32.75,900965,33.00
3,AIE,2023-02-01,2.80,2.82,2.78,786865,2.78
4,AIT,2023-02-01,6.70,6.80,6.70,1242712,6.75


In [18]:
file_name = "Price.csv"
output_file = data_path + file_name
osd_file = osd_path + file_name

df_merge.set_index("Symbol", inplace=True)
df_merge.to_csv(output_file, header=None)
df_merge.to_csv(osd_file, header=True)